In [ ]:
from debugpy.common.log import newline
import csv
import time
import pandas as pd
import ipywidgets as widgets
import cv2
import mediapipe as mp
from IPython.display import display

In [ ]:
dataset = []
frame_id = 0
start_time = time.time()

In [ ]:
videoquelle = 'YTrubberband.mp4' 
name = videoquelle.split(".")[0]    
name

In [ ]:
# Videoquelle: 0 = Webcam, oder Dateipfad einsetzen
cap = cv2.VideoCapture(videoquelle)

In [ ]:
mp_draw = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

Use this cell to run via Webcam

In [ ]:
# cap = cv2.VideoCapture(0)

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:

#     while cap.isOpened():
#         ret, image = cap.read()

#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False
#         results = pose.process(image)
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

#         cv2.imshow("Webcam", image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break 

# cap.release()
# cv2.destroyAllWindows()

To stop the video, press 'q'.
To create a new CSV press 'PACE' 

In [ ]:
csv_counter = 0
key = cv2.waitKey(1) & 0xFF

newCSV_button = 32
stop = ord('q')
videoSpeed = 1000               # ms per frame 

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    timestamp = time.time() - start_time

    if results.pose_landmarks:

        mp_draw.draw_landmarks(
        frame,
        results.pose_landmarks,
        mp.solutions.pose.POSE_CONNECTIONS
        )
        for idx, lm in enumerate(results.pose_landmarks.landmark):
            dataset.append([
                frame_id,
                timestamp,
                idx,
                lm.x,
                lm.y,
                lm.z
            ])
            
    cv2.imshow("Video", frame)

    if key == newCSV_button:
        filename = f"../output/pose_{csv_counter}.csv"

        with open(filename, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["frame","time","joint","x","y","z"])
            writer.writerows(dataset)

        csv_counter += 1

    if cv2.waitKey(videoSpeed) & 0xFF == stop:
        break

    frame_id += 1

In [ ]:
def export_full():
    with open("..\output\pose_full.csv",mode="w",newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame","time","joint","x","y","z"])
        writer.writerows(dataset)

In [ ]:
export_full()
cap.release()
cv2.destroyAllWindows()